In [3]:
import json
import os
from tqdm import tqdm
from dotenv import load_dotenv
import hashlib
import spacy

from elasticsearch import Elasticsearch

load_dotenv()
ES_HOST = os.getenv("ES_HOST", "http://localhost:9200")
INDEX_NAME = os.getenv("INDEX_NAME")

es_client = Elasticsearch('http://localhost:9200')

lemmatize = spacy.load("en_core_web_sm")

In [ ]:
with open("data/putin_complete.json", "r") as f:
    speeches = json.load(f)

try:
    es_client.indices.delete(index=INDEX_NAME)
except:
    print(f"Index {INDEX_NAME} already exists")

mapping = {
    "properties": {
        "id":   {"type": "integer"},
        "unique_hash": {"type": "text"},
        "title": {"type": "text"},
        "text":  {"type": "text"},
        "date": {"type": "date"},
    }
}

es_client.indices.create(
    index=INDEX_NAME,
    mappings=mapping,
)

index = 0

for doc in tqdm(speeches, "Indexing..."):
    subset_speech = {k: doc[k] for k in doc.keys() if k in ["date", "title", "transcript_filtered"]}
    subset_speech["text"] = subset_speech["transcript_filtered"]
    subset_speech.pop("transcript_filtered")
    subset_speech["unique_hash"] = hashlib.sha256(subset_speech["text"].encode()).hexdigest()[:8]
    index += 1
    es_client.index(index=INDEX_NAME, id=index, document=subset_speech)

Indexing...: 100%|██████████| 9838/9838 [01:04<00:00, 152.22it/s]


In [9]:
subset_speech

{'date': '1999-12-31T00:01:00',
 'title': 'New Year Address by Acting President Vladimir Putin',
 'text': 'Dear friends, On New Year’s Eve, my family and I planned to gather round the TV, just as you probably did, to listen to the address by President Boris Yeltsin. But things took a different turn. On December 31, 1999, Russia’s first president decided to resign. He has asked me to address the Russian people today. The powers of the head of state have been turned over to me today. The presidential election will be held in three months. I assure you that there will be no vacuum of power, not for a minute. I promise you that any attempts to act contrary to the Russian law and constitution will be cut short. The state will stand firm to protect the freedom of speech, the freedom of conscience, the freedom of the mass media, ownership rights, these fundamental elements of a civilised society. The Armed Forces, the Federal Frontier Service, and law-enforcement agencies are working in the u

In [14]:
response = es_client.search(
    index=INDEX_NAME,
    query={
            "range": {
                "date": {
                    "gte": "2000-01-01",
                    "lte": "2001-12-31"
                }
            }
        }
)

subset = response["hits"]["hits"]

In [ ]:
index = 0
for doc in subset:
    index += 1
    es_client.index(index="test_index", id=index, document=doc["_source"])

In [16]:
rep = es_client.search(index=INDEX_NAME, query={"match_all": {}}, size=10000)["hits"]["hits"]